In [1]:
from __future__ import annotations

import operator
from typing import TypedDict, List, Annotated

from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

In [2]:
class Task(BaseModel):
    id:int
    title:str
    brief:str=Field(...,description="what to cover")

In [3]:
class Plan(BaseModel):
    blog_title:str
    tasks:List[Task]

In [4]:
class State(TypedDict):
    topic:str
    plan:Plan
    sections:Annotated[List[str],operator.add]
    final:str

In [5]:
llm=ChatGroq(model="groq:openai/gpt-oss-20b")

In [6]:
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, client=<groq.resources.chat.completions.Completions object at 0x000002B649B59E80>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002B649B5A510>, model_name='groq:openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [7]:
def planner(state:State)->dict:
    plan=llm.with_structured_output(Plan).invoke(
        [
            SystemMessage(
                content=("Create a blog plan with 5-7 sections on the following topic.")   
            ),
            HumanMessage(content=f"Topic: {state['topic']}"),
        ]
    )
    return {"plan":plan}

In [ ]:
graph = StateGraph(State)
graph.add_node("orchestrator", planner)
graph.add_node("worker", worker)
graph.add_node("reducer", reducer)

In [ ]:
graph.add_edge(START, "planner")
graph.add_conditional_edges("planner", fanout, ["worker"])
graph.add_edge("worker", "reducer")
graph.add_edge("reducer", END)

app = graph.compile()
app

In [ ]:
out = app.invoke({"topic": "Write a blog on Self Attention", "sections": []})